In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nsl_data_utils.misc.plotter import *
from matplotlib.backends.backend_pdf import PdfPages

In [3]:
from nsl_data_utils.loaders.sp_loader import load_sp_paths, load_sp
from nsl_data_utils.loaders.net_loader import load_net_names
from nsl_data_utils.loaders.constants import (
    ARTIFICIAL_ER,
    ARTIFICIAL_PA,
    ARTIFICIAL_SMALL,
    ARXIV_NETSCIENCE_COAUTHORSHIP,
    AUCS,
    CKM_PHYSICIANS,
    EU_TRANSPORTATION,
    FMRI74,
    L2_COURSE,
    LAZEGA,
    TIMIK1Q2009,
    TOY_NETWORK,
)

all_net_types = [
    ARTIFICIAL_ER,
    ARTIFICIAL_PA,
    ARTIFICIAL_SMALL,
    ARXIV_NETSCIENCE_COAUTHORSHIP,
    AUCS,
    CKM_PHYSICIANS,
    EU_TRANSPORTATION,
    FMRI74,
    L2_COURSE,
    LAZEGA,
    TIMIK1Q2009,
    TOY_NETWORK,
]

FEASIBLE_SLICE = {"OR": [0.05, 0.10, 0.15, 0.20], "AND": [0.80, 0.85, 0.90, 0.95]}
ANALYSED_NETWORK = all_net_types[4] # 3, 4, 5, 6, 8, 9, 11

OUT_ROOT_DIR = "eda"

In [ ]:
for net_type in all_net_types:
    print(net_type)
    net_names = load_net_names(net_type)
    for net_name in net_names:
        sp_paths = load_sp_paths(net_type, net_name)
        print(f"\t{net_name}, {[len(spp) for spp in sp_paths.values()]}")

In [ ]:
net_names_raw_sps = {}
for net_name in load_net_names(ANALYSED_NETWORK):
    sp_paths = load_sp_paths(ANALYSED_NETWORK, net_name)
    print(f"\t{net_name}, {len(sp_paths)}")
    net_names_raw_sps[net_name] = load_sp(sp_paths)

In [ ]:
def prepare_df(raw_csv: dict[str, pd.DataFrame]) -> pd.DataFrame:
    concat_csv = []
    for batch_name, batch_df in raw_csv.items():
        batch_df = batch_df.loc[
            ((batch_df["protocol"] == "AND") & (batch_df["p"].isin(FEASIBLE_SLICE["AND"]))) |
            ((batch_df["protocol"] == "OR") & (batch_df["p"].isin(FEASIBLE_SLICE["OR"])))
        ]
        concat_csv.append(batch_df)
    concat_csv = pd.concat(concat_csv, axis=0, ignore_index=True)
    result_grouped = concat_csv.groupby(by=[NETWORK, PROTOCOL, P, ACTOR])
    result_mean = result_grouped.mean()
    result_std = result_grouped.std()
    result_all = pd.merge(result_mean, result_std, left_index=True, right_index=True, suffixes=("_avg", "_std")).sort_index().reset_index(ACTOR)
    return result_all

net_names_prepared_sps = {net_name: prepare_df(raw_sps) for net_name, raw_sps in net_names_raw_sps.items()}

In [10]:
out_dir = pathlib.Path(f"{OUT_ROOT_DIR}/{ANALYSED_NETWORK}")
out_dir.mkdir(exist_ok=True, parents=True)

In [11]:
def plot_variance(result_all: pd.DataFrame):
    
    network = result_all.reset_index()["network"][0]
    pdf = PdfPages(out_dir / (f"plots_{network}.pdf"))

    # title page - network
    partial_fig = plot_title_page(network)
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    # obtain y axis values' range for this network
    maxx = result_all.loc[network].max()
    ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
    sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
    pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
    pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

    # mean result across the network
    partial_fig = plot_partial_result(
        partial_result=result_all.loc[network].groupby(ACTOR).mean().reset_index(),
        suptitle=r"$\bf{" + f"net-{network}" + "}$",
        sl_range=sl_range,
        ex_range=ex_range,
        pi_range=pi_range,
        pt_range=pt_range,
    )
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    for proto in ["AND", "OR"]:

        # title page - protocol
        partial_fig = plot_title_page(proto)
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # obtain y axis values' range for this network and protocol
        maxx = result_all.loc[network].loc[proto].max()
        ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
        sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
        pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
        pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

        # mean result across the network and protocol
        partial_fig = plot_partial_result(
            partial_result=result_all.loc[network].loc[proto].groupby(ACTOR).mean().reset_index(),
            suptitle=f"net-{network}--" + r"$\bf{" + f"proto-{proto}" + "}$",
            sl_range=sl_range,
            ex_range=ex_range,
            pi_range=pi_range,
            pt_range=pt_range,
        )
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # plot result for all particular experiments
        for p in FEASIBLE_SLICE[proto]:
            partial_fig = plot_partial_result(
                partial_result=result_all.loc[network].loc[proto].loc[p],
                suptitle=f"net-{network}--proto-{proto}--" + r"$\bf{" + f"p-{p}" + "}$",
                sl_range=sl_range,
                ex_range=ex_range,
                pi_range=pi_range,
                pt_range=pt_range,
            )
            partial_fig.savefig(pdf, format="pdf")
            plt.close(partial_fig)

    pdf.close()

In [ ]:
for net_name, prepared_sps in net_names_prepared_sps.items():
    print(net_name)
    plot_variance(prepared_sps)

## Similarity of top actors across various spreading regimes

In [ ]:
def prepare_similarity_df(result_all: pd.DataFrame, net_name: str):
    rankings_raw = {}

    for proto in ["AND", "OR"]:
        _rankings_raw = []
        for p in FEASIBLE_SLICE[proto]:
            _rankings_raw.append(
                list(
                    result_all.loc[net_name].loc[proto].loc[p].sort_values(
                        f"{SIMULATION_LENGTH}_avg", ascending=False
                    )[ACTOR]
                )
            )
        rankings_raw[proto] = np.array(_rankings_raw)

    return rankings_raw

net_names_rankings_raw = {net_name: prepare_similarity_df(prepared_sps, net_name) for net_name, prepared_sps in net_names_prepared_sps.items()}
net_names_rankings_raw

### Kendall's W (doesn't work yet)

In [11]:
# a = np.array(
#     [
#         [5, 6, 9, 1, 3, 0, 8, 2, 4, 7, 10],
#         [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
#         [10, 9, 8, 7, 6, 4, 5, 3, 2, 1, 0],
#     ]
# )
# def prepare_R_matrix(actors_ranking):
#     R = np.zeros_like(actors_ranking)
#     for rankind_idx, ranking in enumerate(actors_ranking):
#         for element_pos, element_id in enumerate(ranking, 1):
#             R[rankind_idx, element_id] = element_pos
    
#     return R

# # prepare_R_matrix(a), a
# rankings_raw[:3, :]

In [12]:
# from typing import Any


# def map_actor_names(raw_rankings: np.ndarray, actor_names: np.array) -> tuple[np.ndarray, dict[Any, int]]:
#     mapping_dict = {actor_name.item(): idx for idx, actor_name in enumerate(actor_names)}
#     mapped_ranking = np.ones_like(rankings_raw) * (rankings_raw.shape[1] - 1)
#     for rr_idx, raw_ranking in enumerate(raw_rankings):
#         for actor_idx, actor_name in enumerate(raw_ranking):
#             mapped_ranking[rr_idx, actor_idx] = mapping_dict[actor_name]
#     return mapped_ranking, mapping_dict


# def prepare_R_matrix(actors_ranking):
#     R = np.zeros_like(actors_ranking)
#     for rankind_idx, ranking in enumerate(actors_ranking):
#         for element_pos, element_id in enumerate(ranking, 1):
#             R[rankind_idx, element_id] = element_pos
#     return R


# def kendalls_W(R_matrix: np.ndarray) -> float:
#     m, n = R_matrix.shape
#     R_total = np.sum(R_matrix, axis=0)
#     R_dash = np.mean(R_total)
#     S = np.sum((R_total - R_dash) ** 2)
#     W = 12 * S / (m**2 * (n**3 - n))
#     return W

In [13]:
# cut_w = []

# for cut in range(1, rankings_raw.shape[1] + 1):
#     cut_rankings = rankings_raw[:, :cut]
#     mapped_ranking, _ = map_actor_names(cut_rankings, np.unique(cut_rankings))
#     R_matrix = prepare_R_matrix(mapped_ranking)
#     W = kendalls_W(R_matrix)
#     cut_w.append(W)

# print(cut_w)


In [14]:
# plt.scatter(x=np.arange(0, len(cut_w)), y=cut_w)

## My metric

In [ ]:
def rankings_raw_to_cuts(rankings_raw: dict[str, np.array]) -> dict[str, int]:

    cuts = {}

    for proto, ranking_raw in rankings_raw.items():

        increments = []
        num_uniques = []

        i_actors = set()
        for j in range(1, ranking_raw.shape[1]):
            j_sample = ranking_raw[:, :j]
            j_actors = set(np.unique(j_sample).tolist())
            increments.append(len(j_actors.difference(i_actors)))
            num_uniques.append(len(j_actors))
            i_actors = j_actors

        num_uniques = 100 * np.array(num_uniques) / len(np.unique(ranking_raw))
        increments = 100 * np.array(increments) / len(np.unique(ranking_raw))

        cuts[proto] = {"num_uniques": num_uniques, "increments": increments}

    return cuts

net_names_cuts = {net_name: rankings_raw_to_cuts(rankings_raw) for net_name, rankings_raw in net_names_rankings_raw.items()}
net_names_cuts

In [ ]:
def plot_cuts(cuts: dict[str, int], net_name: str):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(9, 4.5))

    for ax, proto in zip(axs, cuts):
        ax_1A = ax
        increments = cuts[proto]["increments"]
        num_uniques = cuts[proto]["num_uniques"]

        color = "tab:red"
        ax_1A.set_xlabel("top-k actors")
        ax_1A.set_ylabel("increment of unique actors between cuts", color=color)
        ax_1A.plot(np.arange(len(increments)), increments, color=color)
        ax_1A.tick_params(axis="y", labelcolor=color)

        ax_1B = ax_1A.twinx()

        color = "tab:blue"
        ax_1B.set_ylabel("% of all actors in the cut", color=color) 
        ax_1B.plot(np.arange(len(num_uniques)), num_uniques, color=color)
        ax_1B.tick_params(axis="y", labelcolor=color)
        ax_1B.set_title(f"protocol: {proto}")

    fig.suptitle("Share of the same top-k actors across all spreading regimes")
    fig.tight_layout()
    plt.savefig(out_dir / (f"ranking_{net_name}.pdf"))
    plt.close(fig)


for net_name, cuts in net_names_cuts.items():
    print(cuts)
    plot_cuts(cuts, net_name)